In [ ]:
import numpy as np
import pandas as pd
import time

# --- 1. LOAD DATASET ---
df = pd.read_csv('Housing.csv')

# --- 2. TARGET AND FEATURES ---
y = df['price'].values

feature_m1 = ['area']
features_m2 = ['area', 'bedrooms', 'bathrooms', 'stories', 'parking', 'mainroad']

# Convert categorical 'mainroad' to 1/0
if 'mainroad' in df.columns:
    df['mainroad'] = df['mainroad'].map({'yes': 1, 'no': 0})

# --- 3. STANDARDIZATION FUNCTION ---
def standardize(X):
    return (X - np.mean(X, axis=0)) / np.std(X, axis=0)

# Prepare clean X1 and X2 (Same as Part 1)
X1 = standardize(df[feature_m1].values)
X2 = standardize(df[features_m2].values)

print("Data Loaded and Standardized!")
print(f"X1 shape: {X1.shape}, X2 shape: {X2.shape}, y shape: {y.shape}")

Data Loaded and Standardized!
X1 shape: (545, 1), X2 shape: (545, 6), y shape: (545,)


In [ ]:
# --- 4. NORMAL EQUATION SOLVER ---
def solve_normal_equation(X, y):
    # Direct solution using the formula: theta = (X @ X^T)^-1 @ X @ y
    # Matches the class derivation for X as an (m+1, n) matrix.

    start_time = time.time()
    
    # 1. Get dimensions (n = number of samples)
    # Note: .shape is a property, no parentheses needed
    n, m = X.shape 
    
    # 2. Add bias: Transpose to (m, n) then add row of ones -> (m+1, n)
    X = X.T
    X = np.vstack((np.ones((1, n)), X)) # X is now (m+1, n)

    # 3. Prepare components
    # X_T is (n, m+1)
    X_T = X.T
    y = y.reshape(n, 1)
    
    # 4. Calculate (X @ X^T)
    # This results in an (m+1, m+1) matrix
    XTX = np.dot(X, X_T)
    
    # 5. Compute the Inverse
    # The formula requires the inverse of (X @ X^T)
    XTX_inv = np.linalg.inv(XTX)
    
    # 6. Calculate X @ y
    # Result is (m+1, 1)
    Xy = np.dot(X, y)

    # 7. Final Theta: (X @ X^T)^-1 @ (X @ y)
    theta_opt = np.dot(XTX_inv, Xy)

    end_time = time.time()
    computation_time = end_time - start_time
    
    return theta_opt, computation_time

# --- 5. EXECUTE NORMAL EQUATION ---
theta_ne1, time_ne1 = solve_normal_equation(X1, y)
theta_ne2, time_ne2 = solve_normal_equation(X2, y)

print("\n--- Normal Equation Results ---")
print(f"Model 1 Theta (Intercept, Area): \n{theta_ne1.flatten()}")
print(f"Model 2 Theta: \n{theta_ne2.flatten()}")


--- Normal Equation Results ---
Model 1 Theta (Intercept, Area): 
[4766729.24770642 1001630.47587323]
Model 2 Theta: 
[4766729.24770642  659884.06435776  150716.00874733  577400.63897293
  439704.94512015  295359.32388137  227634.90097247]


In [ ]:
from sklearn.model_selection import train_test_split

# --- 6. TRAIN/TEST SPLIT (80% Train, 20% Test) ---
X1_train, X1_test, y_train, y_test = train_test_split(X1, y, test_size=0.2, random_state=42)
X2_train, X2_test, _, _ = train_test_split(X2, y, test_size=0.2, random_state=42)

# --- 7. HELPER: COMPUTE MSE LOSS ---
def compute_mse(X, y, theta):
    n,m= X.shape
    # X is now m*n
    X=X.T

    # X is now (m+1)*n
    X=np.vstack((np.ones((1,n)),X))

    h= np.dot(X.T,theta)

    y=y.reshape(n,1)
    term= h-y
    term_T=(h-y).T

    error= ( 1/n * (np.dot(term_T,term)) )

    return error.item()

In [ ]:
class LinearRegressionScratch:
    def __init__ (self,eta=0.01,epochs=1000):
        self.eta=eta
        self.epochs=epochs
        self.train_loss=[]
        self.theta=None
        self.theta_history=[]
    
    def compute_loss(self,X,y):
        # X shape: (m+1, n)
        # y shape: (n, 1)
        # self.theta shape: (m+1, 1)
    
        n = X.shape[1]
        h = np.dot(X.T, self.theta) # (n, 1)
        term = h - y.reshape(n, 1)   # (n, 1)
    
        # Loss = 1/n * (term_transpose @ term)
        loss = (1 / n) * np.dot(term.T, term)
    
        return loss.item() # Returns a single number for easy plotting
       

    # X here means X_train, and y means y_train
    def fit(self,X,y):
        # Transpose X to (m, n)
        X = X.T 
        m, n = X.shape
        
        # Add bias row -> X is now (m+1, n)
        X = np.vstack((np.ones((1, n)), X))

        # Ensure y is (n, 1)
        y = y.reshape(n, 1)
        
        # Initialize θ = (m+1, 1)
        self.theta = np.zeros((m + 1, 1))

        for _ in range(self.epochs):
            # h = (n, 1)
            h = np.dot(X.T, self.theta)

            # Gradient = (m+1, 1)
            gradient = (2/ n) * np.dot(X, (h - y))

            # Update
            self.theta -= self.eta * gradient

            # Track
            self.train_loss.append(self.compute_loss(X, y))
            self.theta_history.append(self.theta.copy())

    
    def predict(self,X):
        n,m= X.shape
        # X is now m*n
        X=X.T
        
        # X is now (m+1)*n
        X = np.vstack((np.ones((1, X.shape[1])), X))

        return np.dot(X.T,self.theta).flatten()
        
        
# --- 8. SOLVE USING GRADIENT DESCENT (From Part 1) ---
# We use Model 2 for the main comparison table (6 features)
gd_model = LinearRegressionScratch(eta=0.01, epochs=1000)

start_gd = time.time()
gd_model.fit(X2_train, y_train)
time_gd = time.time() - start_gd

theta_gd = gd_model.theta
train_loss_gd = gd_model.train_loss[-1]
test_loss_gd = compute_mse(X2_test, y_test, theta_gd)

# --- 9. SOLVE USING NORMAL EQUATION ---
theta_ne, time_ne = solve_normal_equation(X2_train, y_train)
train_loss_ne = compute_mse(X2_train, y_train, theta_ne)
test_loss_ne = compute_mse(X2_test, y_test, theta_ne)

In [ ]:
# --- 10. GENERATE COMPARISON TABLE ---
results = {
    "Method": ["Gradient Descent", "Normal Equation"],
    "Train Loss": [f"{train_loss_gd:.2e}", f"{train_loss_ne:.2e}"],
    "Test Loss": [f"{test_loss_gd:.2e}", f"{test_loss_ne:.2e}"],
    "Time Taken (s)": [f"{time_gd:.6f}", f"{time_ne:.6f}"],
    "Remarks": [
        "Iterative, requires tuning eta/epochs.", 
        "Analytical, exact solution, very fast for small datasets."
    ]
}

df_comparison = pd.DataFrame(results)
print("\n--- FINAL COMPARISON TABLE (Model 2) ---")
print(df_comparison.to_string(index=False))

# --- 11. PARAMETER COMPARISON ---
print("\n--- LEARNED PARAMETERS COMPARISON ---")
print(f"GD Theta (first 3): {theta_gd.flatten()[:3]}")
print(f"NE Theta (first 3): {theta_ne.flatten()[:3]}")


--- FINAL COMPARISON TABLE (Model 2) ---
          Method Train Loss Test Loss Time Taken (s)                                                   Remarks
Gradient Descent   1.30e+12  2.25e+12       0.098348                    Iterative, requires tuning eta/epochs.
 Normal Equation   1.30e+12  2.25e+12       0.000190 Analytical, exact solution, very fast for small datasets.

--- LEARNED PARAMETERS COMPARISON ---
GD Theta (first 3): [4744602.73152016  611069.36907268  133733.50104189]
NE Theta (first 3): [4744603.17844871  611070.68709012  133726.77953169]
